In [ ]:
from torch_geometric.datasets import GeometricShapes
from torch_geometric.transforms import SamplePoints, KNNGraph
import torch_geometric.transforms as T
import networkx as nx
from brec.dataset import BRECDataset
from brec.evaluator import evaluate
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
import numpy as np
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from diversity_curves.utils import make_splits
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc
from diversity_curves.utils import make_splits
from sklearn.svm import SVC
import pandas as pd

Processing...
Done!


In [ ]:
gm_train = BRECDataset("../../Data")

In [55]:
# Assuming g6_list is the list of 51,200 graphs generated by your process()
extracted_pairs = []

for i in range(400):
    #for i in range(400,800): ### Reliability Set
    # Each pair i starts at index (i * 64)
    # The first version of Graph A is at that start index
    # The first version of Graph B is at the very next index due to interlacing
    
    idx_a = i * 64
    idx_b = i * 64 + 1
    
    graph_a = gm_train[idx_a]
    graph_b = gm_train[idx_b]
    extracted_pairs.append(graph_a)
    extracted_pairs.append(graph_b)

In [99]:
# Assuming g6_list is the list of 51,200 graphs generated by your process()
extracted_pairs_test = []

for i in range(400, 800):
    #for i in range(400,800): ### Reliability Set
    # Each pair i starts at index (i * 64)
    # The first version of Graph A is at that start index
    # The first version of Graph B is at the very next index due to interlacing
    
    idx_a = i * 64
    idx_b = i * 64 + 1
    
    graph_a = gm_train[idx_a]
    graph_b = gm_train[idx_b]
    extracted_pairs_test.append(graph_a)
    extracted_pairs_test.append(graph_b)

In [56]:
#   "Basic": (0, 60),
#   "Regular": (60, 160),
#   "Extension": (160, 260),
#   "CFI": (260, 360),
#   "4-Vertex_Condition": (360, 380),
#   "Distance_Regular": (380, 400),

In [ ]:
def pyg_dataset_to_networkx(gm_train):
    labels = []
    nx_graphs = []
    for data in gm_train:
        edges_list = data.edge_index.numpy().T.tolist()
        G = nx.Graph(edges_list)
        nx_graphs.append(G)
    return labels, nx_graphs

In [ ]:
labels_train, nx_graphs_train = pyg_dataset_to_networkx(extracted_pairs)
labels_test, nx_graphs_test = pyg_dataset_to_networkx(extracted_pairs_test)
y_train = labels_train 
data_train = nx_graphs_train 
G_nx = nx_graphs_train

In [60]:
n_nodes = [g.number_of_nodes() for g in G_nx]

In [ ]:
import numpy as np
from diversity_curves.diversity import compute_magnitude_graph
from magnipy.magnitude.distances import get_dist

G_nx = nx_graphs_train


for metric in ["shortest_path_distance","heat_kernel_distance","diffusion_distance"]:
    def spread(G):
        def dist_fn(G):
            return get_dist(G=G, X=None, X2=None, metric=metric, mode="structure", check_for_duplicates=False)

        compute_at_cardinality = [G.number_of_nodes()]
        mag0,ts=compute_magnitude_graph(G, dist_fn, ts=[1],method="spread", get_weights=False)
        return mag0
    test_results=[]
    for i in range(0,800,2):
        #print(i)
        j=i+1
        compute_at_cardinality = [n_nodes[i]]
        spread_i = spread(G_nx[i])
        spread_j = spread(G_nx[j])
        
        stat = np.abs(spread_i-spread_j)
        p_value = stat >10**(-5)
        test_results.append(p_value[0])
    test_results_df = pd.DataFrame({"pval": pd.Series(test_results)})
    lables = ["Basic" for r in range(60)]+["Regular" for r in range(100)]+["Extension" for r in range(100)]+["CFI" for r in range(100)]+["4V" for r in range(20)]+["DR" for r in range(20)]
    test_results_df.columns=["pval"]
    test_results_df["lables"]=lables
    test_results_df["sig"] = test_results_df["pval"]#<0.05
    results = test_results_df[["lables", "sig"]].groupby(["lables"]).sum().T
    ordered = results[["Basic", "Regular", "Extension", "CFI", "4V", "DR"]]
    ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
    ordered.loc["prop",:] = (ordered.loc["sig",:] / ordered.loc["all",:]).values
    ordered["sum"] = ordered.sum(axis=1)
    ordered["sum"]["prop"] = ordered["sum"]["sig"] / ordered["sum"][ "all"]
    ordered_v2 = ordered.copy()
    ordered_v2["Regular"] = ordered_v2["Regular"]+ordered_v2["4V"]+ordered_v2["DR"]
    ordered_v2=ordered_v2.drop(columns=["4V", "DR"])
    ordered_v2.loc["prop",:] = (ordered_v2.loc["sig",:] / ordered_v2.loc["all",:]).values
    print(metric)
    print(ordered_v2)
    ordered_v2.to_csv(f"./spread_{metric}.csv")

    bol1 = ~test_results_df.sig
    
    test_results2=[]
    all_mean_spreads=[]
    for i in range(0,800,2):
        j=i+1
        
        ind = int(i/2)
        if bol1[ind]:
            print(ind)
            G1 = G_nx[i].copy()
            G2 = G_nx[j].copy()
            spread_i = []
            for ei in G1.edges():
                g = nx.contracted_edge(G1, ei, self_loops=False, copy=True)
                spread_i.append(spread(g))
            
            
            spread_j = []
            for ei in G2.edges():
                g = nx.contracted_edge(G2, ei, self_loops=False, copy=True)
                spread_j.append(spread(g))

            spread_i = np.mean(spread_i)
            spread_j = np.mean(spread_j)
            
            stat = np.abs(spread_i-spread_j)
            p_value = stat >10**(-5)
            test_results2.append(p_value)
            all_mean_spreads.append([spread_i, spread_j])
    sum(test_results2)
    retry_df = test_results_df[["lables"]].loc[bol1,:]
    retry_df["sig"] = test_results2
    results = retry_df[["lables", "sig"]].groupby(["lables"]).sum().T
    results.to_csv(f"./one_edge_{metric}.csv")
    pd.DataFrame(all_mean_spreads).to_csv(f"./one_edge_{metric}_mean_spreads.csv")

/tmp/ipykernel_415302/3960706824.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/3960706824.py:47: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps up

shortest_path_distance
lables      Basic     Regular  Extension     CFI     sum
sig     16.000000   13.000000      41.00    6.00   76.00
all     60.000000  140.000000     100.00  100.00  400.00
prop     0.266667    0.092857       0.41    0.06    0.19
0
1
2
3
4
5
6
7
8
9
10
11
12
13
16
18
19
22
23
24
25
26
27
29
30
32
33
35
36
39
40
41
42
43
44
45
46
48
49
50
51
52
54
55
60
61
62
63
64
65
66
67
68
69
70
71
72
74
76
77
78
80
81
85
86
89
91
92
93
95
96
99
100
101
102
103
104
105
106
107
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
165
166
170
171
172
173
174
179
180
182
184
185
188
189
190
194
195
197
199
200
201
204
207
208
209
210
211
212
217
218
219
222
228
229
237
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
266
267
268
269
270
271
272
273
274
275
276
277
279

/tmp/ipykernel_415302/3960706824.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/3960706824.py:47: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps up

heat_kernel_distance
lables  Basic     Regular  Extension     CFI       sum
sig      60.0   50.000000      92.00    3.00  205.0000
all      60.0  140.000000     100.00  100.00  400.0000
prop      1.0    0.357143       0.92    0.03    0.5125
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
173
241
243
245
248
249
253
254
260
261
262
263
264
266
267
268
269
270
271
272
273
274
275
276
277
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
394

/tmp/ipykernel_415302/3960706824.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/3960706824.py:47: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps up

diffusion_distance
lables  Basic  Regular  Extension     CFI     sum
sig      60.0    49.00      84.00    3.00  196.00
all      60.0   140.00     100.00  100.00  400.00
prop      1.0     0.35       0.84    0.03    0.49
71
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
173
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
260
261
262
263
264
266
267
268
269
270
271
272
273
274
275
276
277
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
39

In [ ]:
import numpy as np
from magnipy.magnitude.distances import get_dist

G_nx = nx_graphs_test

for metric in ["shortest_path_distance","heat_kernel_distance","diffusion_distance"]:
    def spread(G):
        def dist_fn(G):
            return get_dist(G=G, X=None, X2=None, metric=metric, mode="structure", check_for_duplicates=False)

        compute_at_cardinality = [G.number_of_nodes()]
        mag0,ts=compute_magnitude_graph(G, dist_fn, ts=[1],method="spread", get_weights=False)
        return mag0
    test_results=[]
    for i in range(0,800,2):
        #print(i)
        j=i+1
        compute_at_cardinality = [n_nodes[i]]
        spread_i = spread(G_nx[i])
        spread_j = spread(G_nx[j])

        stat = np.abs(spread_i-spread_j)
        p_value = stat >10**(-5) 
        test_results.append(p_value[0])
        
    test_results_df = pd.DataFrame({"pval": pd.Series(test_results)})
    lables = ["Basic" for r in range(60)]+["Regular" for r in range(100)]+["Extension" for r in range(100)]+["CFI" for r in range(100)]+["4V" for r in range(20)]+["DR" for r in range(20)]
    test_results_df.columns=["pval"]
    test_results_df["lables"]=lables
    test_results_df["sig"] = test_results_df["pval"]#<0.05
    results = test_results_df[["lables", "sig"]].groupby(["lables"]).sum().T
    ordered = results[["Basic", "Regular", "Extension", "CFI", "4V", "DR"]]
    ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
    ordered.loc["prop",:] = (ordered.loc["sig",:] / ordered.loc["all",:]).values
    ordered["sum"] = ordered.sum(axis=1)
    ordered["sum"]["prop"] = ordered["sum"]["sig"] / ordered["sum"][ "all"]
    ordered_v2 = ordered.copy()
    ordered_v2["Regular"] = ordered_v2["Regular"]+ordered_v2["4V"]+ordered_v2["DR"]
    ordered_v2=ordered_v2.drop(columns=["4V", "DR"])
    ordered_v2.loc["prop",:] = (ordered_v2.loc["sig",:] / ordered_v2.loc["all",:]).values
    print(metric)
    print(ordered_v2)

/tmp/ipykernel_415302/210095835.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/210095835.py:46: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps upda

shortest_path_distance
lables  Basic  Regular  Extension    CFI    sum
sig       0.0      0.0        0.0    0.0    0.0
all      60.0    140.0      100.0  100.0  400.0
prop      0.0      0.0        0.0    0.0    0.0


/tmp/ipykernel_415302/210095835.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/210095835.py:46: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps upda

heat_kernel_distance
lables  Basic  Regular  Extension    CFI    sum
sig       0.0      0.0        0.0    0.0    0.0
all      60.0    140.0      100.0  100.0  400.0
prop      0.0      0.0        0.0    0.0    0.0
diffusion_distance
lables  Basic  Regular  Extension    CFI    sum
sig       0.0      0.0        0.0    0.0    0.0
all      60.0    140.0      100.0  100.0  400.0
prop      0.0      0.0        0.0    0.0    0.0


/tmp/ipykernel_415302/210095835.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ordered.loc["all",:] = [60, 100, 100, 100, 20, 20]
/tmp/ipykernel_415302/210095835.py:46: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps upda